# 📄 IntelliCode-SL | Docs + Explanation SLM — 8-bit Quantized

**Model:** `meta-llama/Llama-3.2-1B-Instruct`
**Config:** 8-bit quantized base + full precision adapter (via HuggingFace BitsAndBytes)

> ⚠️ Run in a **fresh Colab session** after 08A and 08B complete.
> Unsloth is intentionally NOT used — loading 8-bit through Unsloth
> causes a `BlockDiagonalCausalMask` conflict on Llama models.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
# NOTE: Unsloth is intentionally NOT imported here.
# Loading 8-bit via Unsloth causes a BlockDiagonalCausalMask conflict.
import os, torch, time, gc, json
from collections import defaultdict
from google.colab import drive
from huggingface_hub import login
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/docs_explanation_adapter"
MODEL_NAME   = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LEN  = 2048
TASKS        = ["explain", "document"]

login(token=HF_TOKEN)
print("✅ Drive mounted + HF login done")
print(f"   Adapter: {ADAPTER_PATH}")


In [ ]:
# ── Cell 4: Prompt Templates ───────────────────────────────────
TASK_PROMPTS = {
    "explain"  : "Explain what the following code does in simple, clear language.",
    "document" : "Generate proper documentation (docstrings, comments, or README) for the following code.",
}

PROMPT_TEMPLATE = """### Task: {task_instruction}

### Code:
{input}

### Response:
"""

print("✅ Prompt templates defined")


In [ ]:
# ── Cell 5: Test Cases (50 per task = 100 total) ───────────────
test_cases = {
    "explain": [
        {"input": "def factorial(n):\n    if n == 0: return 1\n    return n * factorial(n-1)", "expected_contains": "factorial"},
        {"input": "def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a+b\n    return a", "expected_contains": "fibonacci"},
        {"input": "def binary_search(arr, t):\n    l,r = 0,len(arr)-1\n    while l<=r:\n        m=(l+r)//2\n        if arr[m]==t: return m\n        elif arr[m]<t: l=m+1\n        else: r=m-1\n    return -1", "expected_contains": "search"},
        {"input": "def flatten(lst):\n    result=[]\n    for item in lst:\n        if isinstance(item,list): result.extend(flatten(item))\n        else: result.append(item)\n    return result", "expected_contains": "nested"},
        {"input": "def memoize(f):\n    cache={}\n    def wrapper(*args):\n        if args not in cache: cache[args]=f(*args)\n        return cache[args]\n    return wrapper", "expected_contains": "cache"},
        {"input": "def quicksort(arr):\n    if len(arr)<=1: return arr\n    pivot=arr[len(arr)//2]\n    left=[x for x in arr if x<pivot]\n    mid=[x for x in arr if x==pivot]\n    right=[x for x in arr if x>pivot]\n    return quicksort(left)+mid+quicksort(right)", "expected_contains": "sort"},
        {"input": "class Stack:\n    def __init__(self): self.items=[]\n    def push(self,x): self.items.append(x)\n    def pop(self): return self.items.pop()\n    def peek(self): return self.items[-1]\n    def is_empty(self): return len(self.items)==0", "expected_contains": "stack"},
        {"input": "def count_words(text):\n    words=text.lower().split()\n    freq={}\n    for w in words: freq[w]=freq.get(w,0)+1\n    return freq", "expected_contains": "word"},
        {"input": "def is_palindrome(s):\n    s=s.lower().replace(' ','')\n    return s==s[::-1]", "expected_contains": "palindrome"},
        {"input": "def gcd(a,b):\n    while b: a,b=b,a%b\n    return a", "expected_contains": "greatest common"},
        {"input": "def sieve(n):\n    primes=[True]*(n+1)\n    p=2\n    while p*p<=n:\n        if primes[p]:\n            for i in range(p*p,n+1,p): primes[i]=False\n        p+=1\n    return [p for p in range(2,n+1) if primes[p]]", "expected_contains": "prime"},
        {"input": "def merge_sort(arr):\n    if len(arr)<=1: return arr\n    mid=len(arr)//2\n    left=merge_sort(arr[:mid])\n    right=merge_sort(arr[mid:])\n    return merge(left,right)", "expected_contains": "sort"},
        {"input": "def lru_cache(cap):\n    from collections import OrderedDict\n    cache=OrderedDict()\n    def get(k):\n        if k not in cache: return -1\n        cache.move_to_end(k); return cache[k]\n    def put(k,v):\n        if k in cache: cache.move_to_end(k)\n        cache[k]=v\n        if len(cache)>cap: cache.popitem(last=False)\n    return get,put", "expected_contains": "cache"},
        {"input": "def dijkstra(graph, start):\n    import heapq\n    dist={node:float('inf') for node in graph}\n    dist[start]=0\n    pq=[(0,start)]\n    while pq:\n        d,u=heapq.heappop(pq)\n        for v,w in graph[u]:\n            if dist[u]+w<dist[v]:\n                dist[v]=dist[u]+w\n                heapq.heappush(pq,(dist[v],v))\n    return dist", "expected_contains": "shortest path"},
        {"input": "class Node:\n    def __init__(self,val=0,left=None,right=None): self.val=val; self.left=left; self.right=right\ndef inorder(root):\n    return inorder(root.left)+[root.val]+inorder(root.right) if root else []", "expected_contains": "tree"},
        {"input": "def two_sum(nums, target):\n    seen={}\n    for i,n in enumerate(nums):\n        if target-n in seen: return [seen[target-n],i]\n        seen[n]=i", "expected_contains": "sum"},
        {"input": "def rotate_matrix(m):\n    n=len(m)\n    for i in range(n):\n        for j in range(i+1,n): m[i][j],m[j][i]=m[j][i],m[i][j]\n    for row in m: row.reverse()\n    return m", "expected_contains": "rotat"},
        {"input": "def valid_brackets(s):\n    stack=[]\n    mapping={')':'(',']':'[','}':'{'}\n    for c in s:\n        if c in mapping:\n            if not stack or stack[-1]!=mapping[c]: return False\n            stack.pop()\n        else: stack.append(c)\n    return not stack", "expected_contains": "bracket"},
        {"input": "def run_length_encode(s):\n    if not s: return ''\n    result=[]\n    count=1\n    for i in range(1,len(s)):\n        if s[i]==s[i-1]: count+=1\n        else: result.append(f'{count}{s[i-1]}'); count=1\n    result.append(f'{count}{s[-1]}')\n    return ''.join(result)", "expected_contains": "encod"},
        {"input": "def max_subarray(nums):\n    max_sum=cur=nums[0]\n    for n in nums[1:]:\n        cur=max(n,cur+n)\n        max_sum=max(max_sum,cur)\n    return max_sum", "expected_contains": "subarray"},
        {"input": "def coin_change(coins,amount):\n    dp=[float('inf')]*(amount+1)\n    dp[0]=0\n    for c in coins:\n        for x in range(c,amount+1):\n            dp[x]=min(dp[x],dp[x-c]+1)\n    return dp[amount] if dp[amount]!=float('inf') else -1", "expected_contains": "coin"},
        {"input": "def number_of_islands(grid):\n    def dfs(i,j):\n        if i<0 or i>=len(grid) or j<0 or j>=len(grid[0]) or grid[i][j]!='1': return\n        grid[i][j]='0'\n        dfs(i+1,j); dfs(i-1,j); dfs(i,j+1); dfs(i,j-1)\n    count=0\n    for i in range(len(grid)):\n        for j in range(len(grid[0])):\n            if grid[i][j]=='1': dfs(i,j); count+=1\n    return count", "expected_contains": "island"},
        {"input": "def knapsack(weights,values,cap):\n    n=len(weights)\n    dp=[[0]*(cap+1) for _ in range(n+1)]\n    for i in range(1,n+1):\n        for w in range(cap+1):\n            dp[i][w]=dp[i-1][w]\n            if weights[i-1]<=w: dp[i][w]=max(dp[i][w],dp[i-1][w-weights[i-1]]+values[i-1])\n    return dp[n][cap]", "expected_contains": "knapsack"},
        {"input": "def topological_sort(graph):\n    visited=set()\n    stack=[]\n    def dfs(v):\n        visited.add(v)\n        for u in graph.get(v,[]):\n            if u not in visited: dfs(u)\n        stack.append(v)\n    for v in graph:\n        if v not in visited: dfs(v)\n    return stack[::-1]", "expected_contains": "topological"},
        {"input": "def levenshtein(s1,s2):\n    m,n=len(s1),len(s2)\n    dp=[[0]*(n+1) for _ in range(m+1)]\n    for i in range(m+1): dp[i][0]=i\n    for j in range(n+1): dp[0][j]=j\n    for i in range(1,m+1):\n        for j in range(1,n+1):\n            if s1[i-1]==s2[j-1]: dp[i][j]=dp[i-1][j-1]\n            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])\n    return dp[m][n]", "expected_contains": "distance"},
        {"input": "def power_set(lst):\n    result=[[]]\n    for x in lst: result+=[s+[x] for s in result]\n    return result", "expected_contains": "subset"},
        {"input": "def caesar_cipher(text,shift):\n    result=[]\n    for c in text:\n        if c.isalpha():\n            base=ord('A') if c.isupper() else ord('a')\n            result.append(chr((ord(c)-base+shift)%26+base))\n        else: result.append(c)\n    return ''.join(result)", "expected_contains": "cipher"},
        {"input": "def bfs(graph,start):\n    from collections import deque\n    visited=set([start])\n    queue=deque([start])\n    order=[]\n    while queue:\n        v=queue.popleft(); order.append(v)\n        for u in graph[v]:\n            if u not in visited: visited.add(u); queue.append(u)\n    return order", "expected_contains": "breadth"},
        {"input": "def trie_insert(root,word):\n    node=root\n    for c in word:\n        if c not in node: node[c]={}\n        node=node[c]\n    node['#']=True", "expected_contains": "trie"},
        {"input": "def longest_common_subsequence(s1,s2):\n    m,n=len(s1),len(s2)\n    dp=[[0]*(n+1) for _ in range(m+1)]\n    for i in range(1,m+1):\n        for j in range(1,n+1):\n            if s1[i-1]==s2[j-1]: dp[i][j]=dp[i-1][j-1]+1\n            else: dp[i][j]=max(dp[i-1][j],dp[i][j-1])\n    return dp[m][n]", "expected_contains": "subsequence"},
        {"input": "def sliding_window_max(nums,k):\n    from collections import deque\n    dq=deque(); result=[]\n    for i,n in enumerate(nums):\n        while dq and dq[0]<i-k+1: dq.popleft()\n        while dq and nums[dq[-1]]<n: dq.pop()\n        dq.append(i)\n        if i>=k-1: result.append(nums[dq[0]])\n    return result", "expected_contains": "window"},
        {"input": "def subset_sum(nums,target):\n    dp={0}\n    for n in nums: dp={x+n for x in dp}|dp\n    return target in dp", "expected_contains": "sum"},
        {"input": "def matrix_multiply(A,B):\n    rows_A,cols_A=len(A),len(A[0])\n    rows_B,cols_B=len(B),len(B[0])\n    C=[[0]*cols_B for _ in range(rows_A)]\n    for i in range(rows_A):\n        for j in range(cols_B):\n            for k in range(cols_A): C[i][j]+=A[i][k]*B[k][j]\n    return C", "expected_contains": "multipl"},
        {"input": "def word_break(s,wordDict):\n    dp=[False]*(len(s)+1); dp[0]=True\n    for i in range(1,len(s)+1):\n        for w in wordDict:\n            if dp[i-len(w)] and s[i-len(w):i]==w: dp[i]=True; break\n    return dp[-1]", "expected_contains": "word"},
        {"input": "def find_median(nums1,nums2):\n    merged=sorted(nums1+nums2)\n    n=len(merged)\n    return merged[n//2] if n%2 else (merged[n//2-1]+merged[n//2])/2", "expected_contains": "median"},
        {"input": "def group_anagrams(strs):\n    from collections import defaultdict\n    groups=defaultdict(list)\n    for s in strs: groups[tuple(sorted(s))].append(s)\n    return list(groups.values())", "expected_contains": "anagram"},
        {"input": "def zigzag(s,rows):\n    if rows==1: return s\n    fence=['']*(rows); row=0; step=1\n    for c in s:\n        fence[row]+=c\n        if row==0: step=1\n        elif row==rows-1: step=-1\n        row+=step\n    return ''.join(fence)", "expected_contains": "zigzag"},
        {"input": "def first_missing_positive(nums):\n    nums=set(nums); i=1\n    while i in nums: i+=1\n    return i", "expected_contains": "missing"},
        {"input": "def trap_water(height):\n    left,right=0,len(height)-1; left_max=right_max=water=0\n    while left<right:\n        if height[left]<height[right]:\n            if height[left]>=left_max: left_max=height[left]\n            else: water+=left_max-height[left]\n            left+=1\n        else:\n            if height[right]>=right_max: right_max=height[right]\n            else: water+=right_max-height[right]\n            right-=1\n    return water", "expected_contains": "water"},
        {"input": "def jump_game(nums):\n    max_reach=0\n    for i,n in enumerate(nums):\n        if i>max_reach: return False\n        max_reach=max(max_reach,i+n)\n    return True", "expected_contains": "jump"},
        {"input": "def unique_paths(m,n):\n    dp=[[1]*n for _ in range(m)]\n    for i in range(1,m):\n        for j in range(1,n): dp[i][j]=dp[i-1][j]+dp[i][j-1]\n    return dp[-1][-1]", "expected_contains": "path"},
        {"input": "def decode_ways(s):\n    n=len(s); dp=[0]*(n+1); dp[0]=1; dp[1]=0 if s[0]=='0' else 1\n    for i in range(2,n+1):\n        one=int(s[i-1:i]); two=int(s[i-2:i])\n        if 1<=one<=9: dp[i]+=dp[i-1]\n        if 10<=two<=26: dp[i]+=dp[i-2]\n    return dp[n]", "expected_contains": "decod"},
        {"input": "def spiral_order(matrix):\n    result=[]\n    while matrix:\n        result+=matrix.pop(0)\n        matrix=list(zip(*matrix))[::-1]\n    return result", "expected_contains": "spiral"},
        {"input": "def search_rotated(nums,target):\n    l,r=0,len(nums)-1\n    while l<=r:\n        m=(l+r)//2\n        if nums[m]==target: return m\n        if nums[l]<=nums[m]:\n            if nums[l]<=target<nums[m]: r=m-1\n            else: l=m+1\n        else:\n            if nums[m]<target<=nums[r]: l=m+1\n            else: r=m-1\n    return -1", "expected_contains": "search"},
        {"input": "def course_schedule(n,prereqs):\n    from collections import defaultdict,deque\n    graph=defaultdict(list); indegree=[0]*n\n    for a,b in prereqs: graph[b].append(a); indegree[a]+=1\n    q=deque([i for i in range(n) if indegree[i]==0]); count=0\n    while q:\n        c=q.popleft(); count+=1\n        for nb in graph[c]: indegree[nb]-=1\n        if indegree[nb]==0: q.append(nb)\n    return count==n", "expected_contains": "course"},
        {"input": "def partition_equal_subset(nums):\n    total=sum(nums)\n    if total%2: return False\n    target=total//2; dp={0}\n    for n in nums: dp={x+n for x in dp if x+n<=target}|dp\n    return target in dp", "expected_contains": "subset"},
    ],
    "document": [
        {"input": "def factorial(n):\n    if n == 0: return 1\n    return n * factorial(n-1)", "expected_contains": "param"},
        {"input": "def binary_search(arr, target):\n    l, r = 0, len(arr) - 1\n    while l <= r:\n        m = (l + r) // 2\n        if arr[m] == target: return m\n        elif arr[m] < target: l = m + 1\n        else: r = m - 1\n    return -1", "expected_contains": "return"},
        {"input": "class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, x): self.items.append(x)\n    def pop(self): return self.items.pop()\n    def peek(self): return self.items[-1]\n    def is_empty(self): return len(self.items) == 0", "expected_contains": "class"},
        {"input": "def merge_sort(arr):\n    if len(arr) <= 1: return arr\n    mid = len(arr) // 2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    return merge(left, right)", "expected_contains": "param"},
        {"input": "def flatten(lst):\n    result = []\n    for item in lst:\n        if isinstance(item, list): result.extend(flatten(item))\n        else: result.append(item)\n    return result", "expected_contains": "param"},
        {"input": "def count_words(text):\n    words = text.lower().split()\n    freq = {}\n    for w in words: freq[w] = freq.get(w, 0) + 1\n    return freq", "expected_contains": "return"},
        {"input": "def gcd(a, b):\n    while b: a, b = b, a % b\n    return a", "expected_contains": "param"},
        {"input": "def sieve(n):\n    primes = [True] * (n+1)\n    p = 2\n    while p*p <= n:\n        if primes[p]:\n            for i in range(p*p, n+1, p): primes[i] = False\n        p += 1\n    return [p for p in range(2, n+1) if primes[p]]", "expected_contains": "prime"},
        {"input": "def lru_cache_fn(cap):\n    from collections import OrderedDict\n    cache = OrderedDict()\n    def get(k):\n        if k not in cache: return -1\n        cache.move_to_end(k); return cache[k]\n    def put(k, v):\n        if k in cache: cache.move_to_end(k)\n        cache[k] = v\n        if len(cache) > cap: cache.popitem(last=False)\n    return get, put", "expected_contains": "cache"},
        {"input": "def dijkstra(graph, start):\n    import heapq\n    dist = {node: float('inf') for node in graph}\n    dist[start] = 0\n    pq = [(0, start)]\n    while pq:\n        d, u = heapq.heappop(pq)\n        for v, w in graph[u]:\n            if dist[u]+w < dist[v]:\n                dist[v] = dist[u]+w\n                heapq.heappush(pq, (dist[v], v))\n    return dist", "expected_contains": "param"},
        {"input": "def memoize(f):\n    cache = {}\n    def wrapper(*args):\n        if args not in cache: cache[args] = f(*args)\n        return cache[args]\n    return wrapper", "expected_contains": "decorator"},
        {"input": "def quicksort(arr):\n    if len(arr) <= 1: return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]\n    mid = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quicksort(left) + mid + quicksort(right)", "expected_contains": "pivot"},
        {"input": "def two_sum(nums, target):\n    seen = {}\n    for i, n in enumerate(nums):\n        if target - n in seen: return [seen[target-n], i]\n        seen[n] = i", "expected_contains": "param"},
        {"input": "def valid_brackets(s):\n    stack = []\n    mapping = {')': '(', ']': '[', '}': '{'}\n    for c in s:\n        if c in mapping:\n            if not stack or stack[-1] != mapping[c]: return False\n            stack.pop()\n        else: stack.append(c)\n    return not stack", "expected_contains": "return"},
        {"input": "def max_subarray(nums):\n    max_sum = cur = nums[0]\n    for n in nums[1:]:\n        cur = max(n, cur + n)\n        max_sum = max(max_sum, cur)\n    return max_sum", "expected_contains": "param"},
        {"input": "def coin_change(coins, amount):\n    dp = [float('inf')] * (amount+1)\n    dp[0] = 0\n    for c in coins:\n        for x in range(c, amount+1): dp[x] = min(dp[x], dp[x-c]+1)\n    return dp[amount] if dp[amount] != float('inf') else -1", "expected_contains": "param"},
        {"input": "def knapsack(weights, values, cap):\n    n = len(weights)\n    dp = [[0]*(cap+1) for _ in range(n+1)]\n    for i in range(1, n+1):\n        for w in range(cap+1):\n            dp[i][w] = dp[i-1][w]\n            if weights[i-1] <= w: dp[i][w] = max(dp[i][w], dp[i-1][w-weights[i-1]]+values[i-1])\n    return dp[n][cap]", "expected_contains": "param"},
        {"input": "def topological_sort(graph):\n    visited = set(); stack = []\n    def dfs(v):\n        visited.add(v)\n        for u in graph.get(v, []):\n            if u not in visited: dfs(u)\n        stack.append(v)\n    for v in graph:\n        if v not in visited: dfs(v)\n    return stack[::-1]", "expected_contains": "param"},
        {"input": "def levenshtein(s1, s2):\n    m, n = len(s1), len(s2)\n    dp = [[0]*(n+1) for _ in range(m+1)]\n    for i in range(m+1): dp[i][0] = i\n    for j in range(n+1): dp[0][j] = j\n    for i in range(1, m+1):\n        for j in range(1, n+1):\n            if s1[i-1] == s2[j-1]: dp[i][j] = dp[i-1][j-1]\n            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])\n    return dp[m][n]", "expected_contains": "distance"},
        {"input": "def sliding_window_max(nums, k):\n    from collections import deque\n    dq = deque(); result = []\n    for i, n in enumerate(nums):\n        while dq and dq[0] < i-k+1: dq.popleft()\n        while dq and nums[dq[-1]] < n: dq.pop()\n        dq.append(i)\n        if i >= k-1: result.append(nums[dq[0]])\n    return result", "expected_contains": "window"},
        {"input": "def matrix_multiply(A, B):\n    rows_A, cols_A = len(A), len(A[0])\n    cols_B = len(B[0])\n    C = [[0]*cols_B for _ in range(rows_A)]\n    for i in range(rows_A):\n        for j in range(cols_B):\n            for k in range(cols_A): C[i][j] += A[i][k]*B[k][j]\n    return C", "expected_contains": "matrix"},
        {"input": "def group_anagrams(strs):\n    from collections import defaultdict\n    groups = defaultdict(list)\n    for s in strs: groups[tuple(sorted(s))].append(s)\n    return list(groups.values())", "expected_contains": "param"},
        {"input": "def number_of_islands(grid):\n    def dfs(i, j):\n        if i < 0 or i >= len(grid) or j < 0 or j >= len(grid[0]) or grid[i][j] != '1': return\n        grid[i][j] = '0'\n        dfs(i+1,j); dfs(i-1,j); dfs(i,j+1); dfs(i,j-1)\n    count = 0\n    for i in range(len(grid)):\n        for j in range(len(grid[0])):\n            if grid[i][j] == '1': dfs(i,j); count+=1\n    return count", "expected_contains": "island"},
        {"input": "def trap_water(height):\n    left, right = 0, len(height)-1\n    left_max = right_max = water = 0\n    while left < right:\n        if height[left] < height[right]:\n            if height[left] >= left_max: left_max = height[left]\n            else: water += left_max - height[left]\n            left += 1\n        else:\n            if height[right] >= right_max: right_max = height[right]\n            else: water += right_max - height[right]\n            right -= 1\n    return water", "expected_contains": "water"},
        {"input": "def jump_game(nums):\n    max_reach = 0\n    for i, n in enumerate(nums):\n        if i > max_reach: return False\n        max_reach = max(max_reach, i+n)\n    return True", "expected_contains": "param"},
        {"input": "def unique_paths(m, n):\n    dp = [[1]*n for _ in range(m)]\n    for i in range(1, m):\n        for j in range(1, n): dp[i][j] = dp[i-1][j] + dp[i][j-1]\n    return dp[-1][-1]", "expected_contains": "path"},
        {"input": "def spiral_order(matrix):\n    result = []\n    while matrix:\n        result += matrix.pop(0)\n        matrix = list(zip(*matrix))[::-1]\n    return result", "expected_contains": "matrix"},
        {"input": "def search_rotated(nums, target):\n    l, r = 0, len(nums)-1\n    while l <= r:\n        m = (l+r)//2\n        if nums[m] == target: return m\n        if nums[l] <= nums[m]:\n            if nums[l] <= target < nums[m]: r = m-1\n            else: l = m+1\n        else:\n            if nums[m] < target <= nums[r]: l = m+1\n            else: r = m-1\n    return -1", "expected_contains": "rotated"},
        {"input": "def subset_sum(nums, target):\n    dp = {0}\n    for n in nums: dp = {x+n for x in dp} | dp\n    return target in dp", "expected_contains": "param"},
        {"input": "def longest_common_subsequence(s1, s2):\n    m, n = len(s1), len(s2)\n    dp = [[0]*(n+1) for _ in range(m+1)]\n    for i in range(1, m+1):\n        for j in range(1, n+1):\n            if s1[i-1] == s2[j-1]: dp[i][j] = dp[i-1][j-1]+1\n            else: dp[i][j] = max(dp[i-1][j], dp[i][j-1])\n    return dp[m][n]", "expected_contains": "subsequence"},
        {"input": "def power_set(lst):\n    result = [[]]\n    for x in lst: result += [s+[x] for s in result]\n    return result", "expected_contains": "subset"},
        {"input": "def caesar_cipher(text, shift):\n    result = []\n    for c in text:\n        if c.isalpha():\n            base = ord('A') if c.isupper() else ord('a')\n            result.append(chr((ord(c)-base+shift)%26+base))\n        else: result.append(c)\n    return ''.join(result)", "expected_contains": "shift"},
        {"input": "def bfs(graph, start):\n    from collections import deque\n    visited = set([start]); queue = deque([start]); order = []\n    while queue:\n        v = queue.popleft(); order.append(v)\n        for u in graph[v]:\n            if u not in visited: visited.add(u); queue.append(u)\n    return order", "expected_contains": "param"},
        {"input": "def word_break(s, wordDict):\n    n = len(s); dp = [False]*(n+1); dp[0] = True\n    for i in range(1, n+1):\n        for w in wordDict:\n            if dp[i-len(w)] and s[i-len(w):i] == w: dp[i] = True; break\n    return dp[-1]", "expected_contains": "param"},
        {"input": "def find_median(nums1, nums2):\n    merged = sorted(nums1+nums2); n = len(merged)\n    return merged[n//2] if n%2 else (merged[n//2-1]+merged[n//2])/2", "expected_contains": "median"},
        {"input": "def first_missing_positive(nums):\n    nums = set(nums); i = 1\n    while i in nums: i += 1\n    return i", "expected_contains": "param"},
        {"input": "def decode_ways(s):\n    n = len(s); dp = [0]*(n+1); dp[0] = 1; dp[1] = 0 if s[0]=='0' else 1\n    for i in range(2, n+1):\n        one = int(s[i-1:i]); two = int(s[i-2:i])\n        if 1 <= one <= 9: dp[i] += dp[i-1]\n        if 10 <= two <= 26: dp[i] += dp[i-2]\n    return dp[n]", "expected_contains": "decod"},
        {"input": "def course_schedule(n, prereqs):\n    from collections import defaultdict, deque\n    graph = defaultdict(list); indegree = [0]*n\n    for a, b in prereqs: graph[b].append(a); indegree[a] += 1\n    q = deque([i for i in range(n) if indegree[i]==0]); count = 0\n    while q:\n        c = q.popleft(); count += 1\n        for nb in graph[c]: indegree[nb] -= 1\n        if indegree[nb] == 0: q.append(nb)\n    return count == n", "expected_contains": "course"},
        {"input": "def max_product_subarray(nums):\n    max_p = min_p = result = nums[0]\n    for n in nums[1:]:\n        candidates = (n, max_p*n, min_p*n)\n        max_p, min_p = max(candidates), min(candidates)\n        result = max(result, max_p)\n    return result", "expected_contains": "product"},
        {"input": "def rotate_matrix(m):\n    n = len(m)\n    for i in range(n):\n        for j in range(i+1, n): m[i][j], m[j][i] = m[j][i], m[i][j]\n    for row in m: row.reverse()\n    return m", "expected_contains": "rotate"},
        {"input": "def run_length_encode(s):\n    if not s: return ''\n    result = []; count = 1\n    for i in range(1, len(s)):\n        if s[i] == s[i-1]: count += 1\n        else: result.append(f'{count}{s[i-1]}'); count = 1\n    result.append(f'{count}{s[-1]}')\n    return ''.join(result)", "expected_contains": "encod"},
        {"input": "def zigzag(s, rows):\n    if rows == 1: return s\n    fence = [''] * rows; row = 0; step = 1\n    for c in s:\n        fence[row] += c\n        if row == 0: step = 1\n        elif row == rows-1: step = -1\n        row += step\n    return ''.join(fence)", "expected_contains": "row"},
        {"input": "def partition_equal_subset(nums):\n    total = sum(nums)\n    if total % 2: return False\n    target = total // 2; dp = {0}\n    for n in nums: dp = {x+n for x in dp if x+n <= target} | dp\n    return target in dp", "expected_contains": "subset"},
        {"input": "def climbing_stairs(n):\n    if n <= 2: return n\n    a, b = 1, 2\n    for _ in range(3, n+1): a, b = b, a+b\n    return b", "expected_contains": "stair"},
        {"input": "def valid_anagram(s, t):\n    from collections import Counter\n    return Counter(s) == Counter(t)", "expected_contains": "anagram"},
        {"input": "def missing_number(nums):\n    n = len(nums)\n    return n*(n+1)//2 - sum(nums)", "expected_contains": "missing"},
        {"input": "def is_power_of_two(n):\n    return n > 0 and (n & (n-1)) == 0", "expected_contains": "power"},
        {"input": "def reverse_linked_list(head):\n    prev = None\n    while head:\n        nxt = head.next\n        head.next = prev\n        prev = head\n        head = nxt\n    return prev", "expected_contains": "reverse"},
    ]
}

print(f"✅ Test cases ready")
for task, cases in test_cases.items():
    print(f"   {task:<10} : {len(cases)} cases")


In [ ]:
# ── Cell 6: Evaluation Helper ──────────────────────────────────
def run_inference(model, tokenizer, task, input_text, max_new_tokens=300):
    prompt = PROMPT_TEMPLATE.format(
        task_instruction=TASK_PROMPTS[task],
        input=input_text
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

def evaluate(model, tokenizer, config_name):
    print(f"\n{'='*58}")
    print(f"  Evaluating: {config_name}")
    print(f"{'='*58}")

    all_results = []
    per_task = {task: {"pass": 0, "fail": 0, "total": 0, "times": []} for task in TASKS}
    total_time = 0
    idx = 0

    for task, cases in test_cases.items():
        print(f"\n  Running {task} ({len(cases)} cases)...")
        for case in cases:
            t0 = time.time()
            output = run_inference(model, tokenizer, task, case["input"])
            elapsed = time.time() - t0
            total_time += elapsed
            per_task[task]["times"].append(elapsed)
            per_task[task]["total"] += 1

            expected = case.get("expected_contains", "")
            passed = (
                len(output) > 20
                and not output.strip().startswith("###")
                and expected.lower() in output.lower()
            )
            if passed:
                per_task[task]["pass"] += 1
            else:
                per_task[task]["fail"] += 1

            all_results.append({
                "task": task,
                "input": case["input"][:80],
                "expected": expected,
                "output": output[:300],
                "passed": passed,
                "time_ms": elapsed * 1000
            })
            idx += 1
            if idx % 25 == 0:
                total_pass = sum(v["pass"] for v in per_task.values())
                print(f"  [{idx:>3}/100]  running pass rate: {total_pass/idx*100:.1f}%")

    total_pass = sum(v["pass"] for v in per_task.values())
    total_cases = sum(v["total"] for v in per_task.values())
    avg_ms = total_time / total_cases * 1000

    print(f"\n  ✅ Overall Pass Rate : {total_pass/total_cases*100:.2f}% ({total_pass}/{total_cases})")
    print(f"  ⏱  Avg latency       : {avg_ms:.1f} ms/prompt")
    print(f"\n  {'Task':<12} {'Pass':>6} {'Fail':>6} {'Total':>7} {'Rate':>8} {'Avg ms':>8}")
    print(f"  {'-'*50}")
    for task in TASKS:
        s = per_task[task]
        rate = s["pass"] / s["total"] * 100
        avg_t = sum(s["times"]) / len(s["times"]) * 1000
        print(f"  {task:<12} {s['pass']:>6} {s['fail']:>6} {s['total']:>7} {rate:>7.1f}% {avg_t:>7.1f}ms")

    return {
        "config": config_name,
        "pass_rate": total_pass / total_cases * 100,
        "total_pass": total_pass,
        "total": total_cases,
        "avg_time_ms": avg_ms,
        "per_task": {
            task: {
                "pass": per_task[task]["pass"],
                "fail": per_task[task]["fail"],
                "total": per_task[task]["total"],
                "pass_rate": per_task[task]["pass"] / per_task[task]["total"] * 100,
                "avg_time_ms": sum(per_task[task]["times"]) / len(per_task[task]["times"]) * 1000
            } for task in TASKS
        },
        "samples": all_results
    }

print("✅ Evaluation helper defined")


In [ ]:
# ── Cell 7: Load Model — 8-bit via HuggingFace BitsAndBytes ────
print("Loading 8-bit quantized model via HuggingFace BitsAndBytes...")

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, torch_dtype=torch.float16)
model.eval()

print("✅ Model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 8: Run Evaluation ─────────────────────────────────────
results = evaluate(model, tokenizer, "Docs+Explanation SLM — 8-bit quantized (HuggingFace)")

In [ ]:
# ── Cell 9: Sample Output Inspection ──────────────────────────
print("Sample outputs (first 3 per task):\n")
shown = {t: 0 for t in TASKS}
for r in results["samples"]:
    task = r["task"]
    if shown[task] < 3:
        print(f"  [{task.upper()}]")
        print(f"  Input    : {r['input'][:70]}...")
        print(f"  Expected : {r['expected'][:60]}")
        print(f"  Output   : {r['output'][:150]}...")
        print(f"  Passed   : {'✅' if r['passed'] else '❌'} | {r['time_ms']:.0f}ms")
        print()
        shown[task] += 1


In [ ]:
# ── Cell 10: Save Results + Final 3-way Comparison ────────────
save_path = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/docs_slm_8bit.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

assert os.path.exists(save_path), f"❌ Save failed"
print(f"✅ Results saved → {save_path}")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Model unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Load all 3 results for comparison ──────────────────────────
paths = {
    "fp16" : "/content/drive/MyDrive/IntelliCode-SL/benchmarks/docs_slm_fp16.json",
    "4-bit": "/content/drive/MyDrive/IntelliCode-SL/benchmarks/docs_slm_4bit.json",
    "8-bit": save_path,
}

all_res = {}
missing = []
for name, p in paths.items():
    try:
        with open(p) as f: all_res[name] = json.load(f)
    except FileNotFoundError:
        missing.append(name)

if missing:
    print(f"\n⚠️  Missing results for: {missing}")
    print("   Run the corresponding notebooks first.")
else:
    configs = ["fp16", "4-bit", "8-bit"]
    print(f"\n{'='*68}")
    print(f"  FINAL COMPARISON: Docs + Explanation SLM — fp16 vs 4-bit vs 8-bit")
    print(f"{'='*68}")
    print(f"  {'Metric':<28}" + "".join(f"{c:>13}" for c in configs))
    print(f"  {'-'*67}")
    print(f"  {'Overall Pass Rate':<28}" + "".join(f"{all_res[c]['pass_rate']:>12.2f}%" for c in configs))
    print(f"  {'Pass / Total':<28}" + "".join(f"{all_res[c]['total_pass']}/{all_res[c]['total']:>10}" for c in configs))
    print(f"  {'Avg Latency':<28}" + "".join(f"{all_res[c]['avg_time_ms']:>11.1f}ms" for c in configs))
    print(f"\n  Per-Task Pass Rate:")
    print(f"  {'Task':<12}" + "".join(f"{c:>13}" for c in configs))
    print(f"  {'-'*51}")
    for task in TASKS:
        row = f"  {task:<12}"
        for c in configs:
            rate = all_res[c]["per_task"][task]["pass_rate"]
            row += f"{rate:>12.1f}%"
        print(row)
    print(f"\n  Speed comparison (vs fp16):")
    fp16_ms = all_res["fp16"]["avg_time_ms"]
    for c in ["4-bit", "8-bit"]:
        ms = all_res[c]["avg_time_ms"]
        ratio = ms / fp16_ms
        faster = "faster" if ratio < 1 else "slower"
        print(f"  {c:<8}: {abs(1-ratio)*100:.1f}% {faster} than fp16 ({ms:.1f}ms vs {fp16_ms:.1f}ms)")
    print(f"\n{'='*68}")
    best_acc = max(configs, key=lambda c: all_res[c]["pass_rate"])
    best_spd = min(configs, key=lambda c: all_res[c]["avg_time_ms"])
    print(f"  Best accuracy : {best_acc} ({all_res[best_acc]['pass_rate']:.2f}%)")
    print(f"  Fastest       : {best_spd} ({all_res[best_spd]['avg_time_ms']:.1f}ms/prompt)")
    print(f"{'='*68}")
